# Prototype 3: Visual Encoder + Video Pipeline

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, math

## 1. Edge Vision Encoder

In [2]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_ch=3, dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, dim, patch_size, patch_size)
        self.pos = nn.Parameter(torch.randn(1, self.num_patches, dim) * 0.02)
    def forward(self, x):
        x = self.proj(x)
        x = x.reshape(x.shape[0], x.shape[1], -1).transpose(1, 2).contiguous()
        return x + self.pos

class TinyBlock(nn.Module):
    def __init__(self, dim, nh=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, nh, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
    def forward(self, x):
        n1 = self.norm1(x)
        x = x + self.attn(n1, n1, n1)[0]
        return x + self.ffn(self.norm2(x))

class EdgeVisionEncoder(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_ch=3, dim=256, n_layers=2, nh=4):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_ch, dim)
        self.blocks = nn.ModuleList([TinyBlock(dim, nh) for _ in range(n_layers)])
        np = self.patch_embed.num_patches
        self.to_token = nn.Linear(dim * np, dim, 0)
    def forward(self, frames):
        b, t, c, h, w = frames.shape
        x = frames.flatten(0, 1)
        x = self.patch_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = x.reshape(b, t, -1)
        return self.to_token(x)

## 2. Hybrid Core

In [3]:
class LinearAttn(nn.Module):
    def __init__(self, dim, decay=0.99, eps=1e-6):
        super().__init__(); self.decay, self.eps = decay, eps
    def fm(self, x): return F.elu(x) + 1.0
    def forward(self, q, k, v, S=None, z=None):
        qf, kf, vf = self.fm(q), self.fm(k), self.fm(v)
        if S is None:
            S = kf.new_zeros(*q.shape[:2], q.shape[-1], q.shape[-1])
            z = kf.new_zeros(*q.shape[:2], q.shape[-1])
        S = self.decay * S + kf.transpose(-2, -1) @ vf
        z = self.decay * z + kf.sum(-2)
        return (qf @ S) / (qf @ z.unsqueeze(-1)).clamp(min=self.eps), S, z

class KVCompress(nn.Module):
    def __init__(self, dim, hd, m, overlap=False):
        super().__init__(); self.m=m; self.overlap=overlap
        self.W_k=nn.Linear(dim,hd,0); self.W_v=nn.Linear(dim,hd,0); self.W_z=nn.Linear(dim,1,0)
        if overlap:
            self.W_kb=nn.Linear(dim,hd,0); self.W_vb=nn.Linear(dim,hd,0); self.W_zb=nn.Linear(dim,1,0)
    def forward(self, h):
        b,n,d=h.shape; m=self.m; nc=n//m; r=nc*m
        hb=h[:,:r].reshape(b,nc,m,d)
        k=self.W_k(hb); v=self.W_v(hb); z=self.W_z(hb).squeeze(-1)
        if self.overlap and nc>1:
            hs=torch.cat([h[:,:1].expand(-1,m,-1),h[:,:r]],1)[:,:r]
            hs=hs.reshape(b,nc,m,d)
            k=torch.cat([k,self.W_kb(hs)],2); v=torch.cat([v,self.W_vb(hs)],2)
            z=torch.cat([z,self.W_zb(hs).squeeze(-1)],-1)
        w=F.softmax(z,-1)
        return (w.unsqueeze(-1)*k).sum(2), (w.unsqueeze(-1)*v).sum(2)

class CSA(nn.Module):
    def __init__(self,dim,hd,m=4,tk=64,decay=0.99):
        super().__init__(); self.tk=tk
        self.cp=KVCompress(dim,hd,m,1); self.qp=nn.Linear(dim,hd,0)
        self.out=nn.Linear(hd,dim,0); self.la=LinearAttn(hd,decay)
    def forward(self,q,h,S=None,z_s=None):
        k,v=self.cp(h); qk=self.qp(q[:,-1:])
        s=(qk@k.transpose(-2,-1))/math.sqrt(k.shape[-1])
        _,ix=torch.topk(s,min(self.tk,k.shape[1]),-1)
        ix=ix.unsqueeze(-1).expand(-1,-1,-1,k.shape[-1]).squeeze(1)
        k=torch.gather(k,1,ix); v=torch.gather(v,1,ix)
        o,S,z_s=self.la(qk.unsqueeze(1),k.unsqueeze(1),v.unsqueeze(1),S,z_s)
        return self.out(o.squeeze(1)),S,z_s

class HCA(nn.Module):
    def __init__(self,dim,hd,m=128,decay=0.999):
        super().__init__()
        self.cp=KVCompress(dim,hd,m,0); self.out=nn.Linear(hd,dim,0)
        self.qp=nn.Linear(dim,hd,0); self.la=LinearAttn(hd,decay)
    def forward(self,q,h,S=None,z_s=None):
        k,v=self.cp(h); qk=self.qp(q[:,-1:]).unsqueeze(1)
        o,S,z_s=self.la(qk,k.unsqueeze(1),v.unsqueeze(1),S,z_s)
        return self.out(o.squeeze(1)),S,z_s

class SWA(nn.Module):
    def __init__(self,dim,nh=4,win=128):
        super().__init__(); self.win=win; self.nh=nh; self.hd=dim//nh
        self.qp=nn.Linear(dim,dim,0); self.kp=nn.Linear(dim,dim,0)
        self.vp=nn.Linear(dim,dim,0); self.op=nn.Linear(dim,dim,0)
    def forward(self,q,h):
        b,n,d=h.shape; c=h[:,-min(n,self.win):]
        k=self.kp(c).view(b,-1,self.nh,self.hd).transpose(1,2)
        v=self.vp(c).view(b,-1,self.nh,self.hd).transpose(1,2)
        q=self.qp(q[:,-1:]).view(b,1,self.nh,self.hd).transpose(1,2)
        a=F.softmax((q@k.transpose(-2,-1))/math.sqrt(self.hd),-1)
        return self.op((a@v).transpose(1,2).reshape(b,1,d).squeeze(1))

class HybridLayer(nn.Module):
    def __init__(self,dim=256,hd=64,nh=4):
        super().__init__()
        self.csa=CSA(dim,hd); self.hca=HCA(dim,hd); self.swa=SWA(dim,nh)
        self.g=nn.Parameter(torch.ones(3))
        self.norm=nn.LayerNorm(dim)
        self.ffn=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self,q,h,cs=None,cz=None,hs=None,hz=None):
        co,cs,cz=self.csa(q,h,cs,cz); ho,hs,hz=self.hca(q,h,hs,hz)
        so=self.swa(q,h); g=F.softmax(self.g,0)
        return self.ffn(self.norm(q[:,-1:]+g[0]*co+g[1]*ho+g[2]*so))+q[:,-1:],(cs,cz),(hs,hz)

## 3. Video Model

In [4]:
class VideoModel(nn.Module):
    def __init__(self, n_layers=2, dim=256, hd=64, nh=4):
        super().__init__()
        self.vision = EdgeVisionEncoder(img_size=64, patch_size=8, dim=dim, n_layers=2, nh=nh)
        self.layers = nn.ModuleList([HybridLayer(dim, hd, nh) for _ in range(n_layers)])
    def stream(self, frames):
        vis = self.vision(frames)
        n = len(self.layers); cs=[None]*n; cz=[None]*n; hs=[None]*n; hz=[None]*n
        h = vis
        for i, l in enumerate(self.layers):
            h,(cs[i],cz[i]),(hs[i],hz[i]) = l(h, h, cs[i], cz[i], hs[i], hz[i])
        return cs, hs

## 4. Test

In [5]:
m = VideoModel(n_layers=2, dim=128, hd=32, nh=4)
print(f"Params: {sum(p.numel() for p in m.parameters()):,}")

f10 = torch.randn(1, 10, 3, 64, 64)
cs, hs = m.stream(f10)
kv10 = sum(s.element_size()*s.numel() for s in cs+hs)/1024

f100 = torch.randn(1, 100, 3, 64, 64)
cs2, hs2 = m.stream(f100)
kv100 = sum(s.element_size()*s.numel() for s in cs2+hs2)/1024

print(f"10 frames: {kv10:.1f} KB | 100 frames: {kv100:.1f} KB (O(1) KV cache)")

Params: 1,955,718
10 frames: 16.0 KB | 100 frames: 16.0 KB (O(1) KV cache)
